Inspect activations stored using command

```
python save_activations.py \
    --model_name "EleutherAI/pythia-410m" \
    --dataset_name "wikitext" \
    --dataset_config_name "wikitext-2-raw-v1" \
    --layer_idx 12 \
    --activation_type "input" \
    --output_dir "/work/nvme/bbjr/eboix/saved_activations" \
    --device "cuda" \
    --dtype "float16" \
    --activation_dl_batch_size 1024 \
    --text_dl_batch_size 8 \
    --max_length 512 \
    --num_val_act_batches 200 \
    --max_val_text_samples 2000
```

In [1]:
activ_dir = '/work/nvme/bbjr/eboix/saved_activations'
notebook_dir = '/u/eboix/moe_distillation'
import os
import torch
from torch.utils.data import TensorDataset, DataLoader
os.chdir(notebook_dir)

In [2]:
os.listdir(activ_dir)

['wikitext_wikitext-2-raw-v1_EleutherAI_pythia-410m_layer12_actinput_validation_mintok20.pt',
 'wikitext_wikitext-2-raw-v1_EleutherAI_pythia-410m_layer12_actinput_train_mintok20.pt']

In [ ]:
# 1. Load the activation data
train_activ_file = activ_dir + '/wikitext_wikitext-2-raw-v1_EleutherAI_pythia-410m_layer12_actinput_train_mintok20.pt'
val_activ_file = activ_dir + '/wikitext_wikitext-2-raw-v1_EleutherAI_pythia-410m_layer12_actinput_validation_mintok20.pt'
train_activ = torch.load(train_activ_file)
val_activ = torch.load(val_activ_file)

# 2. Wrap into TensorDataset
train_dataset = TensorDataset(train_activ)
val_dataset = TensorDataset(val_activ)

# 3. Create DataLoader with appropriate settings
batch_size = 512 # Batch size can be adjusted; float16 uses less memory
shuffle_data = True
num_workers = 0 # Adjust based on your machine's CPU cores
pin_memory = True # Enable for faster CPU to GPU transfers

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=shuffle_data,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True # Keeps the last batch even if it is smaller
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=shuffle_data,
    num_workers=num_workers,
    pin_memory=pin_memory,
    drop_last=True # Keeps the last batch even if it is smaller
)

# Example of how to iterate and move data to GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("CUDA not available, using CPU. GPU transfer will not occur.")


print(f"Using device: {device}")
print(f"DataLoader configured with batch_size={batch_size}, shuffle={shuffle_data}, num_workers={num_workers}, pin_memory={pin_memory}")

# Training loop example
num_epochs = 5 # For example
for epoch in range(num_epochs):
    for i, batch_of_activations_tuple in enumerate(train_dataloader):
        # Since TensorDataset wraps a single tensor (activations),
        # batch_of_activations_tuple will be a tuple with one element.
        batch_of_activations = batch_of_activations_tuple[0]

        # Move batch to GPU
        # non_blocking=True can be used with pin_memory=True for potential overlap
        # between data transfer and GPU computation.
        batch_of_activations = batch_of_activations.to(device, non_blocking=True if pin_memory else False)

        # --- Your training code here ---
        # optimizer.zero_grad()
        # outputs = model(batch_of_activations) # Ensure model can handle input type (float16)
        # loss = criterion(outputs, ...)
        # loss.backward()
        # optimizer.step()

        # For demonstration, just print the shape, device, and dtype of the first batch
        if 'batch_of_activations' in locals() and i == 0: # Check if batch_of_activations exists and it's the first batch
           print(f"Batch shape: {batch_of_activations.shape}, Device: {batch_of_activations.device}, Dtype: {batch_of_activations.dtype}")
           # break # Remove this break for actual training across all batches
    # if 'batch_of_activations' in locals() and epoch == 0: # Stop after the first epoch for demonstration purposes
    #    break

Using device: cuda
DataLoader configured with batch_size=2048, shuffle=True, num_workers=0, pin_memory=True
Batch shape: torch.Size([2048, 1024]), Device: cuda:0, Dtype: torch.float16
Batch shape: torch.Size([2048, 1024]), Device: cuda:0, Dtype: torch.float16


In [ ]:
!nvidia-smi